# SpaNCy-Shift Stage 2 — a genuinely spatial GNN (region graph) vs a per-cell control

**Why:** the published Stage 2 builds its k-NN per `scene_id`, but `scene_id` restarts for every
sample (scene000 = 12 samples with overlapping x, y), its edges point the wrong way and its sampler
scatters cells (~0.32 of 15 neighbours seen) — so it is graph-based in name only (no-graph ≈
old-graph). `spancy_shift_region.py` = `spancy_shift_graphfix.py` (neighbour→cell edges, spatial
patch sampler, 2-hop halo at training and inference) **+ one change: every spatial grouping uses
the region `sample_id|scene_id` (75 regions)**. Each cell sees its own 15 neighbours, own patient.

**Arms, seeds 6/7/8, published config (10 ep, α=0.6, w_recon 0.1, w_contrast 0.5, w_mmd 1.0):**
- `region` — region graph, w_adv 0.3 (candidate graph-based method)
- `percell` — same network, no message passing (`graph_mode='none'`), w_adv 0.3 (control)
- `region-noadv` — as `region`, w_adv 0.0 (is the adversarial term needed?)

`region` and `percell` are also inferred at **α matched** to the published correction size
(mean unimodal |log1p change| 0.0539; inference only). The published model is not retrained: its
numbers are reused (kBET 0.709 ± 0.006, silhouette 0.360 ± 0.002, 0 distorted markers).

**Order (kBET ≈ 20 min per layer, so it runs last and only where needed):**
§1a pre-flight asserts (stops before any training if an assumption on sample / scene / batch is
wrong) → §1b graph audit → §3 training (~2.5 h) → §3b–§6 cheap metrics → **§7 gate** → §8 kBET on
gate-passing layers only (≤ 12 layers, usually ≤ 9) → **§9 verdict**.

**Pre-set rules (decision on seeds 6–8; no re-reading of borderline cases):**
- **Gate** (needed to get kBET): silhouette ≥ 0.355 every seed, mean distorted markers = 0, bimodal
  delta ≤ 0.001, no divergence.
- **A — region usable as the method:** `region` passes the gate AND mean kBET ≥ 0.709 − 0.016.
- **B — the graph adds structure:** kBET(region@αm) − kBET(percell@αm) > 0 every seed AND
  silhouette(region@αm) ≥ silhouette(percell@αm) − 0.005 every seed.
- **C — drop the adversarial term:** `region-noadv` passes the gate, mean kBET ≥ `region` − 0.016,
  and its kBET SD over seeds ≤ 2 × `region`'s.
- **α fallback (§7b):** only if `region` fails the gate on shape and/or silhouette (not divergence,
  not bimodal markers), the same models are re-inferred at α = 0.5, then 0.4; the first passing α
  replaces `region` in rule A (rule C then not tested).

**Neighbourhood check:** after each `region` / `region-noadv` training, the share of GATv2
attention each cell puts on its 15 neighbours (vs itself) is printed (uniform = 0.94, 0 = ignored).

**To run:** Runtime → Run all (GPU). Upload `spancy_shift.py` (published, used only by the §1b
audit) and `spancy_shift_region.py`. **Paste back §1a, §1b, §7, §7b and §9.**

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload the two modules (requested only if missing).
from google.colab import files
import os

NEEDED = ['spancy_shift.py', 'spancy_shift_region.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')
assert 'def region_labels' in open('spancy_shift_region.py').read(), 'wrong spancy_shift_region.py (no region_labels)'
print('Region module: OK')

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from scipy.stats import iqr as _iqr
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift as ssp            # published module: §1b audit only
import spancy_shift_region as sr     # region-graph module: all training / inference
from spancy_shift_region import (
    load_adata, detect_bimodal_markers, find_best_sample_per_marker,
    positive_population_table,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
bim_idx = [marker_names.index(m) for m in bimodal_names]
print(f'{len(uni_markers)} unimodal markers; bimodal: {bimodal_names}')

## 1a. Pre-flight — sample / scene / batch assumptions (asserts, ~2 min, before any training)

In [ ]:
o = adata.obs
N = adata.n_obs
REG = sr.region_labels(adata)
samp = pd.Categorical(o['sample_id'].astype(str)).codes
bat = pd.Categorical(o['batch_id'].astype(str)).codes

# 1. counts
counts = dict(samples=o['sample_id'].nunique(), batches=o['batch_id'].nunique(),
              scene_labels=o['scene_id'].nunique(), regions=len(np.unique(REG)))
print('1. counts:', counts)
assert counts == dict(samples=20, batches=7, scene_labels=10, regions=75), 'unexpected counts'

# 2. nesting: region -> one sample (by construction) and one batch; sample -> one batch
sb = o.groupby('sample_id', observed=True)['batch_id'].nunique()
if sb.max() != 1:
    print(pd.crosstab(o['sample_id'], o['batch_id']).to_string())
    raise AssertionError('STOP: a sample spans several batches — Stage 1 per-sample shift and the '
                         'region grouping need rethinking before training')
rb = pd.Series(o['batch_id'].astype(str).values).groupby(REG).nunique()
assert rb.max() == 1, 'a region spans several batches'
print('2. every sample (and region) lies in exactly one batch:')
print('   ' + ', '.join(f'{s}:{b}' for s, b in
                        o.groupby('sample_id', observed=True)['batch_id'].first().astype(str).items()))

# 3. coordinates are region-local (informational: a region's box starts near 0)
xy = o[['x', 'y']].astype(float)
mins = xy.groupby(REG).min()
print(f'3. region x_min max {mins.x.max():.0f}, y_min max {mins.y.max():.0f}; '
      f'regions with x_min and y_min < 3000: {int(((mins.x < 3000) & (mins.y < 3000)).sum())}/75')

# 4. region graph: neighbours only from the same sample and batch
knn_reg = sr.build_knn_graphs(adata, k=15)
cross_s = float((samp[knn_reg] != samp[:, None]).mean())
cross_b = float((bat[knn_reg] != bat[:, None]).mean())
self_ent = float((knn_reg == np.arange(N)[:, None]).mean())
print(f'4. region graph: cross-sample {100 * cross_s:.3f}%  cross-batch {100 * cross_b:.3f}%  '
      f'self/padding entries {100 * self_ent:.3f}%')
assert cross_s == 0 and cross_b == 0, 'region graph links different samples/batches'

# 5. one training step as train() builds it: all batches present, ~15 neighbours per target
batch_codes_t = o['batch_id'].astype('category').cat.codes.values.astype(np.int64)
scene_codes_t = pd.Categorical(REG).codes.astype(np.int64)
ps = sr.PatchSampler(batch_codes_t, scene_codes_t, xy.values.astype(np.float32), n_per_batch=512, n_patches=4)
adj_reg = sr.AdjacencyIndex(knn_reg)
idx = ps.sample()
ctx = adj_reg.expand(idx, hops=2)
deg = adj_reg.target_in_degree(idx, adj_reg.get_subgraph(ctx))
print(f'5. one PatchSampler step: {len(idx)} targets, {len(np.unique(batch_codes_t[idx]))} batches, '
      f'mean target in-degree {deg:.2f} (full kNN = 15)')
assert len(np.unique(batch_codes_t[idx])) == 7 and deg > 12, 'training step does not see the graph'

# 6. kBET groups exist
GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}
for g, pairs in GROUP_DEFS.items():
    for s, b in pairs:
        n = int(((o['sample_id'] == s) & (o['batch_id'] == b)).sum())
        assert n > 0, f'kBET group {g}: no cells for ({s}, {b})'
print('6. kBET groups: all (sample, batch) subsets non-empty')
print('\nPRE-FLIGHT PASSED')
del ps, idx, ctx

## 1b. Graph audit — does the PUBLISHED graph really link patients, and are those links used?

Counts directed neighbour links of the published graph (`spancy_shift.build_knn_graphs`, per
`scene_id`) at three levels: (i) the neighbour lists, (ii) links present in the mini-batches of one
training epoch (published `SceneBasedSampler`, seed 42 — a link is only used when both cells are in
the batch), (iii) links present in the 2048-cell inference chunks of `spancy_shift.normalize_adata`.

In [ ]:
t0 = time.time()
knn_pub = ssp.build_knn_graphs(adata, k=15)
adj_pub = ssp.AdjacencyIndex(knn_pub)
rows = []


def _row(name, src, dst, n_cells):
    rows.append(dict(level=name, links=len(src), links_per_cell=len(src) / max(n_cells, 1),
                     pct_cross_sample=100 * float((samp[src] != samp[dst]).mean()) if len(src) else 0.0,
                     pct_cross_batch=100 * float((bat[src] != bat[dst]).mean()) if len(src) else 0.0))


cells = np.repeat(np.arange(N), knn_pub.shape[1])
_row('published: neighbour lists', cells, knn_pub.ravel(), N)
cells_r = np.repeat(np.arange(N), knn_reg.shape[1])
_row('region: neighbour lists', cells_r, knn_reg.ravel(), N)
del cells, cells_r

# (ii) training, one epoch, exactly as spancy_shift.train() samples
scene_col = 'scene_id'
scene_codes_pub = o[scene_col].astype('category').cat.codes.values.astype(np.int64)
smp = ssp.SceneBasedSampler(batch_codes_t, scene_codes_pub, n_per_batch=512)
src_l, dst_l, n_seen = [], [], 0
for _ in range(len(smp)):
    idx = smp.sample()
    e = adj_pub.get_subgraph(idx)
    src_l.append(idx[e[0]]); dst_l.append(idx[e[1]]); n_seen += len(idx)
_row('published: used in training (1 epoch)', np.concatenate(src_l), np.concatenate(dst_l), n_seen)

# (iii) inference, exactly as spancy_shift.normalize_adata() chunks
scenes = o[scene_col].values
src_l, dst_l = [], []
for sc in np.unique(scenes):
    scene_idx = np.where(scenes == sc)[0]
    for start in range(0, len(scene_idx), 2048):
        ch = scene_idx[start:start + 2048]
        e = adj_pub.get_subgraph(ch)
        src_l.append(ch[e[0]]); dst_l.append(ch[e[1]])
_row('published: used at inference', np.concatenate(src_l), np.concatenate(dst_l), N)
del src_l, dst_l, knn_pub, adj_pub, smp
gc.collect()

audit_df = pd.DataFrame(rows)
print(audit_df.round(3).to_string(index=False))
print(f'({time.time() - t0:.0f} s)')

## 3. Train `region`, `percell`, `region-noadv` (seeds 6–8) + α-matched inference

α matching: the target is the published correction size (mean unimodal |log1p change| = 0.0539).
If an arm is already within ±3 % at α = 0.6 its 0.6 layer is reused; otherwise α is rescaled once
and refined with one secant step. Achieved ratios are printed.

In [ ]:
SEEDS = [6, 7, 8]
KBET_SEEDS = SEEDS
POP_SEEDS = [6]
K_NEIGHBORS = 15
PUB = dict(kbet=0.709, kbet_tol=0.016, sil=0.360, sil_tol=0.005, uni_delta=0.0539)
SIL_BAR = PUB['sil'] - PUB['sil_tol']
MATCH_TOL = 0.03

BASE = dict(
    n_epochs=10, lr=1e-3, warmup_epochs=2, n_per_batch=512, n_patches=4, k_neighbors=K_NEIGHBORS,
    hybrid_alpha=0.6, temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
    mmd_samples=256, grl_max=1.0,
)
ARMS = {'region': dict(graph_mode='fixed'),
        'percell': dict(graph_mode='none'),
        'region-noadv': dict(graph_mode='fixed', w_adv=0.0)}
MATCH_ARMS = ['region', 'percell']
KEY = {'region': 'region', 'percell': 'percell', 'region-noadv': 'regnoadv'}


def lkey(arm, seed, matched=False):
    return f'{KEY[arm]}{"_am" if matched else ""}_s{seed}'


adata_norm = None
base_log = None
LAYERS = [('normalized_base', 'Stage 1', None, None)]
MATCHED = {}          # (arm, seed) -> (layer key, alpha, achieved ratio)
MODELS = {}           # ('region', seed) -> (model on CPU, scaler, ref) for the §7b alpha fallback
ATT = {}              # (arm, seed) -> neighbour share of attention per GAT layer
train_info = {}
ATT_CELLS = 2000


def attention_share(m, sc, seed):
    # Share of GATv2 attention that each target cell puts on its 15 neighbours (1 - self-loop
    # weight), averaged over ATT_CELLS random cells and over heads. Uniform attention over
    # self + 15 neighbours would give 15/16 = 0.94; 0 = the cell ignores its neighbourhood.
    rng = np.random.default_rng(1000 + seed)
    tgt = rng.choice(N, ATT_CELLS, replace=False)
    ctx = adj_reg.expand(tgt, hops=2)
    e = torch.tensor(adj_reg.get_subgraph(ctx), dtype=torch.long, device=DEVICE)
    xb = np.log1p(np.clip(np.asarray(adata_norm.layers['normalized_base'])[ctx], 0, None))
    x = torch.tensor(sc.transform(xb).astype(np.float32), device=DEVICE)
    enc = m.encoder.to(DEVICE).eval()
    with torch.no_grad():
        h = enc.act(enc.proj(x))
        h1, (ei1, a1) = enc.conv1(h, e, return_attention_weights=True)
        h = enc.act(enc.norm1(h1 + h))
        _, (ei2, a2) = enc.conv2(h, e, return_attention_weights=True)
    out = {}
    for name, ei, a in [('layer1', ei1, a1), ('layer2', ei2, a2)]:
        a = (a.mean(dim=1) if a.dim() > 1 else a).cpu().numpy()
        src, dst = ei[0].cpu().numpy(), ei[1].cpu().numpy()
        is_self = (src == dst) & (dst < len(tgt))
        self_w = np.zeros(len(tgt))
        np.add.at(self_w, dst[is_self], a[is_self])
        out[name] = float(1 - self_w.mean())
    return out


def _uni_delta(g):
    return float(np.abs(np.log1p(np.clip(g[:, uni_idx], 0, None)) - base_log[:, uni_idx]).mean())


def _infer(m, sc, ref, alpha, mode):
    global adata_norm, base_log
    torch.manual_seed(100)
    torch.cuda.manual_seed_all(100)
    out = sr.normalize_adata(adata, m, sc, ref, hybrid_alpha=alpha, k_neighbors=K_NEIGHBORS,
                             inference_batch_size=2048, device_str=DEVICE,
                             layer_name='normalized', keep_base_layer=True, graph_mode=mode)
    g = np.asarray(out.layers['normalized'], dtype=np.float32)
    if adata_norm is None:
        adata_norm = out
        del adata_norm.layers['normalized']
        base_log = np.log1p(np.clip(np.asarray(adata_norm.layers['normalized_base']), 0, None)).astype(np.float32)
    else:
        del out
    gc.collect()
    return g


t0 = time.time()
for seed in SEEDS:
    for arm, over in ARMS.items():
        cfg = {**BASE, **over}
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        t_arm = time.time()
        m, sc, ref, hist = sr.train(
            adata, device_str=DEVICE, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
            ref_sample_per_marker=ref_sample_per_marker, **cfg)
        train_min = (time.time() - t_arm) / 60
        mode = cfg['graph_mode']
        g06 = _infer(m, sc, ref, 0.6, mode)
        k06 = lkey(arm, seed)
        adata_norm.layers[k06] = g06
        LAYERS.append((k06, f'{arm} s{seed}', arm, seed))
        d06 = _uni_delta(g06)
        if mode == 'fixed':
            ATT[(arm, seed)] = attention_share(m, sc, seed)
        if arm == 'region':
            MODELS[(arm, seed)] = (m.cpu(), sc, ref)
        if arm in MATCH_ARMS:
            target = PUB['uni_delta']
            if abs(d06 / target - 1) <= MATCH_TOL:
                MATCHED[(arm, seed)] = (k06, 0.6, d06 / target)
            else:
                a1 = 0.6 * target / d06
                g1 = _infer(m, sc, ref, a1, mode)
                d1 = _uni_delta(g1)
                a, g, d = a1, g1, d1
                if abs(d1 / target - 1) > 0.01 and d1 != d06:
                    a2 = a1 + (target - d1) * (a1 - 0.6) / (d1 - d06)
                    g = _infer(m, sc, ref, a2, mode)
                    a, d = a2, _uni_delta(g)
                    del g1
                km = lkey(arm, seed, matched=True)
                adata_norm.layers[km] = g
                LAYERS.append((km, f'{arm}@am s{seed}', f'{arm}@am', seed))
                MATCHED[(arm, seed)] = (km, a, d / target)
                del g
        train_info[k06] = dict(train_min=train_min, uni_delta_06=d06)
        del g06, m, sc, ref, hist
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        msg = f'  seed {seed}  {arm:12s} train {train_min:.1f} min  uni_delta@0.6 {d06:.4f}'
        if (arm, seed) in ATT:
            msg += '  attention on neighbours ' + ' '.join(f'{k} {v:.2f}' for k, v in ATT[(arm, seed)].items())
        if (arm, seed) in MATCHED:
            km, a, r = MATCHED[(arm, seed)]
            msg += f'  matched alpha {a:.3f} (ratio {r:.3f})'
        print(msg + f'  (total {(time.time() - t0) / 60:.1f} min)')

print(f'\n{len(LAYERS)} layers: {[l[1] for l in LAYERS]}')

In [ ]:
# 3b. Divergence check — run after §3, before §4.
# Without the Huber term (no-Huber, all-off) nothing bounds the size of the correction, and the
# expm1 at inference overflows (max 4e10 / inf). Such layers break the metric cells (inf -> histogram
# and silhouette errors), so they are reported here and dropped from LAYERS: their verdict is
# "diverged". Criterion: any non-finite value, or > 0.1% of entries above 1.2x the raw per-marker
# log1p max (the same 1.2x cap rule as SpaNCy-GNN).
CAP = 1.2
raw_max = np.log1p(np.clip(X_raw, 0, None).max(axis=0))
diverged, div_rows = {}, []
for key, label, arm, seed in LAYERS:
    X = np.asarray(adata_norm.layers[key])
    with np.errstate(over='ignore', invalid='ignore'):
        X_log = np.log1p(np.clip(X, 0, None))
    finite = np.isfinite(X_log)
    over = ~finite | (X_log > CAP * raw_max)
    frac_over = over.mean(axis=0)
    bad_markers = [f'{marker_names[i]} {100 * frac_over[i]:.1f}%'
                   for i in np.argsort(-frac_over) if frac_over[i] > 0.001]
    is_div = (~finite).any() or over.mean() > 0.001
    if is_div and arm is not None:
        diverged[key] = (arm, seed)
    div_rows.append(dict(layer=label, nonfinite=int((~finite).sum()),
                         pct_over_cap=round(100 * float(over.mean()), 3),
                         max_log1p=round(float(X_log[finite].max()), 2),
                         diverged=is_div, markers_over_cap=', '.join(bad_markers[:6])))
    del X, X_log, finite, over
    gc.collect()

print(f'Raw log1p max over markers: {raw_max.max():.2f}  (cap = {CAP}x per marker)')
print(pd.DataFrame(div_rows).to_string(index=False))
LAYERS = [l for l in LAYERS if l[0] not in diverged]
print(f'\nDropped {len(diverged)} diverged layers from the metric cells: '
      f'{sorted(f"{a} s{s}" for a, s in diverged.values())}')

## 4. 1D shape + correction on bimodal markers

Same per-(marker, sample) ratios as cell 6b of the other notebooks (`_shape_ratios` copied verbatim),
but computed for **all** markers so the bimodal ones (ECAD, …) are visible. `bim_delta` = mean
|log1p(arm) − log1p(Stage 1)| over bimodal markers: ~0 means bimodal markers were left alone.

In [ ]:
N_SHAPE_BINS = 80
MIN_CELLS = 50
PEAK_LO, VAR_LO, VAR_HI = 0.80, 0.80, 1.25

sample_ids_shape = adata_norm.obs['sample_id'].values
sample_idx = {s: np.where(sample_ids_shape == s)[0] for s in np.unique(sample_ids_shape)}
sample_idx = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


shape_rows, shape_sum = [], {}
for key, label, arm, seed in LAYERS:
    X_log = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        ps, vs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(X_raw_log[idx, mi], X_log[idx, mi])
            if np.isfinite(p):  ps.append(p)
            if np.isfinite(v):  vs.append(v)
            if np.isfinite(ir): irs.append(ir)
        shape_rows.append(dict(key=key, marker=mname, peak=np.mean(ps), var=np.mean(vs), iqr=np.mean(irs)))
    df = pd.DataFrame([r for r in shape_rows if r['key'] == key]).set_index('marker')
    bad = df[(df.peak < PEAK_LO) | (df['var'] < VAR_LO) | (df['var'] > VAR_HI)]
    u = df.loc[uni_markers]
    shape_sum[key] = dict(
        peak=float(u.peak.mean()), var=float(u['var'].mean()), iqr=float(u.iqr.mean()),
        dist=len(bad), dist_list=', '.join(bad.index),
        ecad_var=float(df.loc['ECAD', 'var']) if 'ECAD' in df.index else np.nan,
        ecad_iqr=float(df.loc['ECAD', 'iqr']) if 'ECAD' in df.index else np.nan,
        bim_delta=float(np.abs(X_log[:, bim_idx] - base_log[:, bim_idx]).mean()) if bim_idx else 0.0,
        uni_delta=float(np.abs(X_log[:, uni_idx] - base_log[:, uni_idx]).mean()),
    )
    del X_log
    gc.collect()
shape_df = pd.DataFrame(shape_rows)

print('Shape summary (peak/var/iqr = unimodal means; dist = ALL markers incl. bimodal):')
print(pd.DataFrame(shape_sum).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).round(4).to_string())

## 5. Silhouette (all seeds)

In [ ]:
# Silhouette — UniFORM-style, 3 cell types, PER SAMPLE (same method + guard as the other notebooks).
N_SIL = 3000
MIN_TYPE_CELLS = 50

anchor_thresholds = {}
for mname in ['ECAD', 'CD45', 'aSMA']:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(X_log_preview[:, mi].reshape(-1, 1))
    anchor_thresholds[mname] = float(np.mean(gmm.means_))
print('Anchor GMM thresholds (log1p, raw):', {k: round(v, 3) for k, v in anchor_thresholds.items()})

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']] = 2
cell_type[X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']] = 1
cell_type[X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']] = 0

# Per-sample selection is decided once (deterministic guard, rng 42 per sample as before).
sample_ids_all = adata_norm.obs['sample_id'].values
sil_sel, dropped = {}, []
for sample in sorted(np.unique(sample_ids_all)):
    smask_idx = np.where(sample_ids_all == sample)[0]
    s_labels = cell_type[smask_idx]
    big = [c for c in (0, 1, 2) if int((s_labels == c).sum()) >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sil_sel[sample] = (smask_idx[sel], s_labels[sel])

sil_rows = {}
for key in ['raw'] + [k for k, _, _, _ in LAYERS]:
    X = X_raw if key == 'raw' else np.asarray(adata_norm.layers[key])
    sil_rows[key] = {s: float(silhouette_score(np.log1p(np.clip(X[ii], 0, None)), lab, metric='euclidean'))
                     for s, (ii, lab) in sil_sel.items()}
sil_df = pd.DataFrame(sil_rows)
sil_df.loc['Mean'] = sil_df.mean()
print(f'{len(sil_sel)} samples passed the guard (dropped: {dropped})')
print(sil_df.loc[['Mean']].round(4).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).to_string())

## 6. Positive population (`POP_SEEDS`)

In [ ]:
pop_sum = {}
for key, label, arm, seed in LAYERS:
    if seed is not None and seed not in POP_SEEDS:
        continue
    pt = positive_population_table(adata_norm, norm_layer=key, sample_col='sample_id')
    s = pt.groupby('marker')['delta'].mean()
    pop_sum[key] = dict(pass_5pct=int((s.abs() < 5).sum()),
                        mean_abs_delta=float(pt['delta'].abs().mean()))
    print(f'  {label:16s} {pop_sum[key]["pass_5pct"]}/20 within +/-5%   '
          f'mean |delta| {pop_sum[key]["mean_abs_delta"]:.2f}%')

## 7. Gate — which layers get kBET (pre-set)

A variant goes to kBET only if, at every seed, silhouette ≥ 0.355 (published − 0.005), and over
the seeds mean distorted markers = 0, bimodal delta ≤ 0.001 and no divergence. §7b then chooses
the kBET layers per rule: A needs `region` (or its α fallback); B needs `region@αm` and
`percell@αm` (only if the region variant passes); C needs `region-noadv` (only without fallback).

In [ ]:
FB_KEYS = {}   # alpha-fallback variants (§7b), e.g. 'region-a0.5' -> {seed: key}


def _keys(variant):
    # layer key per seed for a variant: 'region', 'percell', 'region-noadv', 'region@am', 'percell@am'
    if variant in FB_KEYS:
        return FB_KEYS[variant]
    if variant.endswith('@am'):
        a = variant[:-3]
        return {s: MATCHED[(a, s)][0] for s in SEEDS if (a, s) in MATCHED}
    return {s: lkey(variant, s) for s in SEEDS}


def _sil(k):
    return float(sil_df.loc['Mean', k]) if k in sil_df.columns else np.nan


GATE_OK, GATE_ROWS = {}, {}


def _gate(v):
    ks = _keys(v)
    div = any(k in diverged for k in ks.values())
    sils = [_sil(k) for k in ks.values() if k not in diverged]
    dist = [shape_sum[k]['dist'] for k in ks.values() if k in shape_sum]
    bim = [shape_sum[k]['bim_delta'] for k in ks.values() if k in shape_sum]
    ok = (not div and len(sils) == len(SEEDS) and len(dist) == len(SEEDS)
          and min(sils) >= SIL_BAR and float(np.mean(dist)) == 0 and max(bim) <= 0.001)
    GATE_OK[v] = ok
    GATE_ROWS[v] = dict(variant=v, sil_min=min(sils) if sils else np.nan,
                        sil_mean=np.mean(sils) if sils else np.nan,
                        dist_mean=np.mean(dist) if dist else np.nan,
                        bim_max=max(bim) if bim else np.nan, diverged=div, gate=ok)
    return ok


for v in ['region', 'region@am', 'percell@am', 'region-noadv', 'percell']:
    _gate(v)
print(pd.DataFrame(list(GATE_ROWS.values())).round(4).to_string(index=False))

## 7b. α fallback (pre-set before any result)

Only if `region` fails the gate on shape and/or silhouette — not on divergence or bimodal markers —
the same trained `region` models are re-inferred at α = 0.5, then α = 0.4 (inference only). The first
α that passes the gate replaces `region` in rule A and goes to kBET. Lower α moves the output toward
Stage 1, so this is a strength dial, not a new model. Rule C is not tested if the fallback is used
(`region-noadv` was inferred at 0.6).

In [ ]:
def _shape_one(key):
    X_log = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    rows = []
    for mi, mname in enumerate(marker_names):
        ps, vs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(X_raw_log[idx, mi], X_log[idx, mi])
            if np.isfinite(p):  ps.append(p)
            if np.isfinite(v):  vs.append(v)
            if np.isfinite(ir): irs.append(ir)
        rows.append(dict(key=key, marker=mname, peak=np.mean(ps), var=np.mean(vs), iqr=np.mean(irs)))
    df = pd.DataFrame(rows).set_index('marker')
    bad = df[(df.peak < PEAK_LO) | (df['var'] < VAR_LO) | (df['var'] > VAR_HI)]
    u = df.loc[uni_markers]
    shape_sum[key] = dict(
        peak=float(u.peak.mean()), var=float(u['var'].mean()), iqr=float(u.iqr.mean()),
        dist=len(bad), dist_list=', '.join(bad.index),
        ecad_var=float(df.loc['ECAD', 'var']) if 'ECAD' in df.index else np.nan,
        ecad_iqr=float(df.loc['ECAD', 'iqr']) if 'ECAD' in df.index else np.nan,
        bim_delta=float(np.abs(X_log[:, bim_idx] - base_log[:, bim_idx]).mean()) if bim_idx else 0.0,
        uni_delta=float(np.abs(X_log[:, uni_idx] - base_log[:, uni_idx]).mean()),
    )


def _sil_one(key):
    X = np.asarray(adata_norm.layers[key])
    vals = {s: float(silhouette_score(np.log1p(np.clip(X[ii], 0, None)), lab, metric='euclidean'))
            for s, (ii, lab) in sil_sel.items()}
    col = pd.Series(vals)
    col['Mean'] = col.mean()
    sil_df[key] = col


REGION_V = 'region'
r_row = GATE_ROWS['region']
fallback_due = (not GATE_OK['region'] and not r_row['diverged']
                and (r_row['bim_max'] <= 0.001))
if fallback_due:
    for a in [0.5, 0.4]:
        v = f'region-a{a}'
        FB_KEYS[v] = {}
        for s in SEEDS:
            m, sc, ref = MODELS[('region', s)]
            k = f'region_a{int(round(a * 10))}_s{s}'
            adata_norm.layers[k] = _infer(m, sc, ref, a, 'fixed')
            LAYERS.append((k, f'{v} s{s}', v, s))
            FB_KEYS[v][s] = k
            _shape_one(k)
            _sil_one(k)
        ok = _gate(v)
        print(f'{v}: gate {ok}   ' + str({kk: (round(vv, 4) if isinstance(vv, float) else vv)
                                           for kk, vv in GATE_ROWS[v].items()}))
        if ok:
            REGION_V = v
            break
else:
    print('No fallback needed' if GATE_OK['region'] else
          'Fallback not applicable (region diverged or touched bimodal markers)')

kbet_variants = []
if GATE_OK[REGION_V]:
    kbet_variants.append(REGION_V)
    if GATE_OK['region@am'] and GATE_OK['percell@am']:
        kbet_variants += ['region@am', 'percell@am']
    if REGION_V == 'region' and GATE_OK['region-noadv']:
        kbet_variants.append('region-noadv')
KBET_LAYERS = []
for v in kbet_variants:
    for k in _keys(v).values():
        if k not in KBET_LAYERS:
            KBET_LAYERS.append(k)
print(f'\nRegion variant for rule A: {REGION_V} (gate {GATE_OK[REGION_V]})')
print(f'kBET variants: {kbet_variants}')
print(f'kBET layers ({len(KBET_LAYERS)}, ~{20 * len(KBET_LAYERS)} min): {KBET_LAYERS}')
if not GATE_OK[REGION_V]:
    print('region fails the gate at every alpha tried -> verdict "a real spatial graph damages biology"; '
          'no kBET needed for it (§9).')

## 8. kBET — only the gate-passing layers (≈ 20 min per layer; can run overnight)

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

# GROUP_DEFS defined and checked in §1a. Stage 1 kBET is deterministic (0.6202) -> skipped.
ABL_LABEL = {k: l for k, l, _, _ in LAYERS}
EVAL_LAYERS = KBET_LAYERS
print(f'kBET on {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

## 9. Verdict (paste this)

In [ ]:
def _kbet(k):
    if k not in all_kbet:
        return np.nan
    kv = [x['kBET'] for x in all_kbet[k].values() if not np.isnan(x['kBET'])]
    return float(np.mean(kv)) if kv else np.nan


recs = []
variants = ['region'] + list(FB_KEYS) + ['region@am', 'percell', 'percell@am', 'region-noadv']
for v in variants:
    for s, k in _keys(v).items():
        if v.endswith('@am'):
            a = MATCHED[(v[:-3], s)][1]
        elif v in FB_KEYS:
            a = float(v.split('-a')[1])
        else:
            a = 0.6
        r = dict(variant=v, seed=s, alpha=a, kBET=_kbet(k), sil=_sil(k))
        if k in shape_sum:
            r.update(dist=shape_sum[k]['dist'], uni_delta=shape_sum[k]['uni_delta'],
                     bim_delta=shape_sum[k]['bim_delta'], var=shape_sum[k]['var'], peak=shape_sum[k]['peak'])
        if k in pop_sum:
            r['pos_pop'] = pop_sum[k]['pass_5pct']
        recs.append(r)
res = pd.DataFrame(recs)
print('=' * 100)
print(f'SEEDS {SEEDS}: per variant (published reference: kBET {PUB["kbet"]} +/- 0.006, '
      f'silhouette {PUB["sil"]} +/- 0.002, 0 distorted, uni_delta {PUB["uni_delta"]})')
print('=' * 100)
cols = ['kBET', 'sil', 'dist', 'uni_delta', 'var', 'peak', 'alpha']
print(res.groupby('variant', sort=False)[cols].agg(['mean', 'std']).round(4).to_string())
print('\nPer seed:')
print(res.round(4).to_string(index=False))
print('\nGraph audit (§1b):')
print(audit_df.round(3).to_string(index=False))
print('\nAttention on neighbours (1 - self-loop weight; uniform over self + 15 = 0.94, 0 = ignores neighbours):')
for (arm, s), d in sorted(ATT.items()):
    print(f'  {arm:12s} s{s}  ' + '  '.join(f'{k} {v:.3f}' for k, v in d.items()))

print('\nPer-group kBET:')
for k in KBET_LAYERS:
    print(f'  {ABL_LABEL[k]:22s} ' + '  '.join(f'{g} {x["kBET"]:.3f}' for g, x in all_kbet.get(k, {}).items()))

R = res.set_index(['variant', 'seed'])
print('\n' + '=' * 100)
print('RULES')
print('=' * 100)
kb_reg = [R.loc[(REGION_V, s), 'kBET'] for s in SEEDS]
A = bool(GATE_OK[REGION_V] and not np.isnan(kb_reg).any() and np.mean(kb_reg) >= PUB['kbet'] - PUB['kbet_tol'])
print(f'  [{"x" if A else " "}] A  {REGION_V} usable: gate {GATE_OK[REGION_V]}, '
      f'kBET mean {np.nanmean(kb_reg):.4f} vs bar {PUB["kbet"] - PUB["kbet_tol"]:.3f}')

B = None
if 'region@am' in kbet_variants:
    dk = [R.loc[('region@am', s), 'kBET'] - R.loc[('percell@am', s), 'kBET'] for s in SEEDS]
    dsl = [R.loc[('region@am', s), 'sil'] - R.loc[('percell@am', s), 'sil'] for s in SEEDS]
    B = bool(all(d > 0 for d in dk) and all(d >= -0.005 for d in dsl))
    print(f'  [{"x" if B else " "}] B  graph adds structure (size-matched): kBET diff '
          + ' '.join(f'{d:+.4f}' for d in dk) + '   silhouette diff ' + ' '.join(f'{d:+.4f}' for d in dsl))
else:
    print('  [ ] B  not tested (region or a matched layer failed the gate)')

C = None
if 'region-noadv' in kbet_variants:
    kb_na = [R.loc[('region-noadv', s), 'kBET'] for s in SEEDS]
    C = bool(np.mean(kb_na) >= np.mean(kb_reg) - PUB['kbet_tol']
             and np.std(kb_na, ddof=1) <= 2 * np.std(kb_reg, ddof=1))
    print(f'  [{"x" if C else " "}] C  drop adversarial: kBET {np.mean(kb_na):.4f} +/- {np.std(kb_na, ddof=1):.4f} '
          f'vs region {np.mean(kb_reg):.4f} +/- {np.std(kb_reg, ddof=1):.4f}')
else:
    print(f'  [ ] C  not tested (region-noadv gate {GATE_OK["region-noadv"]}, region variant {REGION_V})')

print('\nVERDICT:')
if A and B:
    print(f'  -> article method = REGION GNN ({REGION_V}); spatial claim allowed (graph beats per-cell at matched size)')
elif A:
    print(f'  -> region GNN ({REGION_V}) usable as the method, but NO claim that the graph helps (B failed / not tested)')
else:
    print('  -> keep the current model; graph-based approach reported as investigated (A failed)')
if C:
    print('  -> adversarial term can be dropped (w_adv = 0)')
elif C is False:
    print('  -> keep the adversarial term (w_adv = 0.3)')